# Modelo Yeast9

In [1]:
from cobra.io import read_sbml_model
from cobra.flux_analysis import flux_variability_analysis
import numpy as np

In [2]:
model_yeast9 = read_sbml_model("yeast-GEM.xml")

In [3]:
# Imprimir información básica del modelo
print("Reacciones:", len(model_yeast9.reactions))
print("Reacciones de exchange:", len(model_yeast9.exchanges))
print("Metabolitos:", len(model_yeast9.metabolites))
print("Genes:", len(model_yeast9.genes))

Reacciones: 4131
Reacciones de exchange: 271
Metabolitos: 2806
Genes: 1161


In [4]:
contador_exportaciones = 0
contador_importaciones = 0
for rxn in model_yeast9.exchanges:
    if rxn.lower_bound < 0:  # Solo reacciones de importación
        contador_importaciones += 1
        print(f"{rxn.id}: {rxn.name} (límite: {rxn.lower_bound})")
    if rxn.upper_bound > 0:  # Solo reacciones de exportación
        contador_exportaciones += 1
        
print(f"Total de reacciones de importación: {contador_importaciones}")
print(f"Total de reacciones de exportación: {contador_exportaciones}")

r_1654: ammonium exchange (límite: -1000.0)
r_1714: D-glucose exchange (límite: -1.0)
r_1832: H+ exchange (límite: -1000.0)
r_1861: iron(2+) exchange (límite: -1000.0)
r_1992: oxygen exchange (límite: -1000.0)
r_2005: phosphate exchange (límite: -1000.0)
r_2020: potassium exchange (límite: -1000.0)
r_2049: sodium exchange (límite: -1000.0)
r_2060: sulphate exchange (límite: -1000.0)
r_2100: water exchange (límite: -1000.0)
r_4593: chloride exchange (límite: -1000.0)
r_4594: Cu2(+) exchange (límite: -1000.0)
r_4595: Mn(2+) exchange (límite: -1000.0)
r_4596: Zn(2+) exchange (límite: -1000.0)
r_4597: Mg(2+) exchange (límite: -1000.0)
r_4600: Ca(2+) exchange (límite: -1000.0)
Total de reacciones de importación: 16
Total de reacciones de exportación: 270


In [15]:
def imprimir_precios_sombra(solucion, model):
    for met_id, value in solucion.shadow_prices.items():
        if abs(value) > 1:
            met = model.metabolites.get_by_id(met_id)
            print(f"{met_id} | {met.name} | shadow price = {value:.3f}")
            
            
def imprimir_costos_reducidos(solucion, model):
    print("Reacciones con costos reducidos positivos")
    # Imprimir resultados
    for rxn_id, value in solucion.reduced_costs.items():
        if value > 1e-6 and model.reactions.get_by_id(rxn_id) in model.exchanges:
            rxn = model.reactions.get_by_id(rxn_id)
            print(f"{rxn_id} | {rxn.name} | shadow price = {value:.3f}")
            
    print("\nReacciones con costos reducidos negativos")
    # Imprimir resultados
    for rxn_id, value in solucion.reduced_costs.items():
        if value < -1 and model.reactions.get_by_id(rxn_id) in model.exchanges:
            rxn = model_yeast9.reactions.get_by_id(rxn_id)
            print(f"{rxn_id} | {rxn.name} | shadow price = {value:.3f}")

Los precios sombra son por metabolito. 

Si el precio sombra de un metabolito es alto (en valor absoluto), significa que: 
- Ese metabolito es limitante para el crecimiento. 
- Una pequeña disponibilidad extra mejoraría la biomasa.

Los costos reducidos son cantidades asociadas a cada reacción (flujo) que indican cuánto tendría que mejorar esa reacción en términos del objetivo (por ejemplo, biomasa) para que pase de estar inactiva (flujo cero) a activarse en la solución óptima. 

En términos prácticos, si estás maximizando crecimiento y una reacción tiene reduced cost positivo:
-  activarla aumentaría la biomasa, pero actualmente está impedida por alguna restricción (como un bound cerrado).

si es negativo:
- activarla disminuiría la biomasa y por eso el modelo no la usa.

y si es cero:
- la reacción ya es óptima o indiferente en la solución actual. 

En resumen, los reduced costs te dicen qué reacciones “querrían” activarse para mejorar el objetivo y cuáles están correctamente apagadas bajo las condiciones actuales.

## FBA modelo tal cual:

In [23]:
with model_yeast9:
    sol1 = model_yeast9.optimize()
    print(f"Solución optimización: {sol1.objective_value}")
    fva_res = flux_variability_analysis(model_yeast9, model_yeast9.exchanges, fraction_of_optimum=0.9)
    imprimir_precios_sombra(sol1, model_yeast9)
    print("\n -----------------------")
    imprimir_costos_reducidos(sol1, model_yeast9)
    

Solución optimización: 0.08584393709068852
s_0045 | (S)-3-hydroxyhexacosanoyl-CoA | shadow price = -1.091
s_0243 | 3-oxohexacosanoyl-CoA | shadow price = -1.085
s_0672 | ergosterol ester backbone | shadow price = -1.386
s_0816 | hexacosanoyl-CoA | shadow price = -1.086
s_0817 | hexacosanoyl-CoA | shadow price = -1.086
s_0819 | hexacosanoyl-CoA | shadow price = -1.091
s_0894 | inositol-P-ceramide A (C24) | shadow price = -1.004
s_0895 | inositol-P-ceramide A (C24) | shadow price = -1.004
s_0897 | inositol-P-ceramide A (C26) | shadow price = -1.039
s_0898 | inositol-P-ceramide A (C26) | shadow price = -1.039
s_0900 | inositol-P-ceramide B (C24) | shadow price = -1.004
s_0901 | inositol-P-ceramide B (C24) | shadow price = -1.004
s_0903 | inositol-P-ceramide B (C26) | shadow price = -1.039
s_0904 | inositol-P-ceramide B (C26) | shadow price = -1.039
s_1479 | tetracosanoyl-CoA | shadow price = -1.051
s_1480 | tetracosanoyl-CoA | shadow price = -1.051
s_1482 | tetracosanoyl-CoA | shadow pric

In [24]:
for rxn_id, fila in fva_res.iterrows():
    if fila["minimum"] < 0:
        rxn = model_yeast9.reactions.get_by_id(rxn_id)
        print(f"{rxn.id} - {rxn.name}: mínimo de {fila['minimum']:.5f}")

r_1654 - ammonium exchange: mínimo de -1.23407
r_1714 - D-glucose exchange: mínimo de -1.00000
r_1832 - H+ exchange: mínimo de -2.23659
r_1861 - iron(2+) exchange: mínimo de -0.00879
r_1992 - oxygen exchange: mínimo de -2.69345
r_2005 - phosphate exchange: mínimo de -4.81593
r_2020 - potassium exchange: mínimo de -0.00031
r_2049 - sodium exchange: mínimo de -0.00034
r_2060 - sulphate exchange: mínimo de -0.44012
r_4593 - chloride exchange: mínimo de -0.00011
r_4594 - Cu2(+) exchange: mínimo de -0.00006
r_4595 - Mn(2+) exchange: mínimo de -0.00023
r_4596 - Zn(2+) exchange: mínimo de -0.00006
r_4597 - Mg(2+) exchange: mínimo de -0.00011
r_4600 - Ca(2+) exchange: mínimo de -0.00002


## FBA modelo con glucosa en -10

In [25]:
with model_yeast9:
    model_yeast9.exchanges.get_by_id("r_1714").lower_bound = -10.0
    sol2 = model_yeast9.optimize()
    print(f"Solución optimización: {sol2.objective_value}")
    fva_res2 = flux_variability_analysis(model_yeast9, model_yeast9.exchanges, fraction_of_optimum=0.9)
    imprimir_precios_sombra(sol2, model_yeast9)
    print("\n -----------------------")
    imprimir_costos_reducidos(sol2, model_yeast9)

Solución optimización: 0.8876853585594556
s_0045 | (S)-3-hydroxyhexacosanoyl-CoA | shadow price = -1.091
s_0243 | 3-oxohexacosanoyl-CoA | shadow price = -1.085
s_0672 | ergosterol ester backbone | shadow price = -1.386
s_0816 | hexacosanoyl-CoA | shadow price = -1.086
s_0817 | hexacosanoyl-CoA | shadow price = -1.086
s_0819 | hexacosanoyl-CoA | shadow price = -1.091
s_0894 | inositol-P-ceramide A (C24) | shadow price = -1.004
s_0895 | inositol-P-ceramide A (C24) | shadow price = -1.004
s_0897 | inositol-P-ceramide A (C26) | shadow price = -1.039
s_0898 | inositol-P-ceramide A (C26) | shadow price = -1.039
s_0900 | inositol-P-ceramide B (C24) | shadow price = -1.004
s_0901 | inositol-P-ceramide B (C24) | shadow price = -1.004
s_0903 | inositol-P-ceramide B (C26) | shadow price = -1.039
s_0904 | inositol-P-ceramide B (C26) | shadow price = -1.039
s_1479 | tetracosanoyl-CoA | shadow price = -1.051
s_1480 | tetracosanoyl-CoA | shadow price = -1.051
s_1482 | tetracosanoyl-CoA | shadow price

In [26]:
for rxn_id, fila in fva_res2.iterrows():
    if fila["minimum"] < 0:
        rxn = model_yeast9.reactions.get_by_id(rxn_id)
        print(f"{rxn.id} - {rxn.name}: mínimo de {fila['minimum']:.5f}")

r_1654 - ammonium exchange: mínimo de -12.76111
r_1714 - D-glucose exchange: mínimo de -10.00000
r_1832 - H+ exchange: mínimo de -23.12780
r_1861 - iron(2+) exchange: mínimo de -0.09087
r_1992 - oxygen exchange: mínimo de -25.80804
r_2005 - phosphate exchange: mínimo de -49.80001
r_2020 - potassium exchange: mínimo de -0.00322
r_2049 - sodium exchange: mínimo de -0.00352
r_2060 - sulphate exchange: mínimo de -4.55114
r_4593 - chloride exchange: mínimo de -0.00115
r_4594 - Cu2(+) exchange: mínimo de -0.00058
r_4595 - Mn(2+) exchange: mínimo de -0.00242
r_4596 - Zn(2+) exchange: mínimo de -0.00066
r_4597 - Mg(2+) exchange: mínimo de -0.00110
r_4600 - Ca(2+) exchange: mínimo de -0.00019


## FBA modelo anaeróbico:

In [4]:
model = read_sbml_model("yeast-GEM.xml")

In [3]:
from modificacion_yeast9 import modify_yeast9_anaerobic

In [5]:
model_anaerobic = modify_yeast9_anaerobic(model)

Removed from r_4598: ['s_3714', 's_1198', 's_1203', 's_1207', 's_1212', 's_0529']
Not found: []


In [7]:
model_anaerobic.exchanges.get_by_id("r_1992").lower_bound = 0.0

In [16]:
with model_anaerobic:
    sol3 = model_anaerobic.optimize()
    print(f"Solución optimización: {sol3.objective_value}")
    fva_res3 = flux_variability_analysis(model_anaerobic, model_anaerobic.exchanges, fraction_of_optimum=0.9)
    imprimir_precios_sombra(sol3, model_anaerobic)
    print("\n -----------------------")
    imprimir_costos_reducidos(sol3, model_anaerobic)

Solución optimización: 0.0
s_0417 | aminoacetaldehyde | shadow price = -10.721
s_0837 | hydrogen peroxide | shadow price = -5.361
s_0839 | hydrogen peroxide | shadow price = -5.361
s_0840 | hydrogen peroxide | shadow price = -5.361
s_1096 | lipid | shadow price = -1.000
s_1275 | oxygen | shadow price = -10.721
s_1276 | oxygen | shadow price = -10.721
s_1277 | oxygen | shadow price = -10.721
s_1278 | oxygen | shadow price = -10.721
s_1279 | oxygen | shadow price = -10.721
s_1293 | palmitoleate | shadow price = -10.721
s_1295 | palmitoleate | shadow price = -10.721
s_1298 | palmitoleate | shadow price = -10.721
s_1392 | pyridoxal | shadow price = -5.361
s_1396 | pyridoxine | shadow price = -5.361
s_1397 | pyridoxine | shadow price = -5.361
s_2817 | oxygen | shadow price = -10.721
s_2819 | palmitoleoyl-CoA(4-) | shadow price = -10.721
s_2837 | palmitoleate | shadow price = -10.721
s_2848 | palmitoleate | shadow price = -10.721
s_2849 | palmitoleoyl-CoA(4-) | shadow price = -10.721
s_2854 

NameError: name 'model_yeast9' is not defined

In [13]:
for rxn_id, fila in fva_res3.iterrows():
    if fila["minimum"] < 0:
        rxn = model_anaerobic.reactions.get_by_id(rxn_id)
        print(f"{rxn.id} - {rxn.name}: mínimo de {fila['minimum']:.5f}")

r_1654 - ammonium exchange: mínimo de -1.30000
r_1714 - D-glucose exchange: mínimo de -1.00000
r_1832 - H+ exchange: mínimo de -1.30000
r_2005 - phosphate exchange: mínimo de -2.60000
r_2060 - sulphate exchange: mínimo de -0.45882
r_2100 - water exchange: mínimo de -0.35455
